# KWISMO — Notebook 01 : Analyse Exploratoire & Visualisations Avancées (EDA)

Ce notebook effectue l'analyse statistique et la visualisation graphique complète du jeu de données KWISMO.

### Visualisations & Compatibilité Multi-Environnements :
- Graphiques dynamiques Matplotlib/Seaborn (distributions, fréquences N-grams, sources, types de médias et catégories).
- Fonctionne en **Local**, sur **Google Colab** et sur **Kaggle Notebooks**.

In [ ]:
# 1. Détection de l'Environnement, Montage Drive & Résolution du Dataset
import os
import sys
import json
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive
        print("Connexion automatique à Google Drive...")
        drive.mount('/content/drive')
    except Exception as err:
        print(f"Montage manuel recommandé : {err}")

current_dir = Path.cwd()
repo_root = current_dir
for candidate in [current_dir, current_dir.parent, current_dir.parent.parent, Path('/content/kwismo/kwismo-ai'), Path('/kaggle/working/kwismo/kwismo-ai')]:
    if (candidate / "data").exists() or (candidate / "src").exists():
        repo_root = candidate.resolve()
        break

local_site_pkgs = repo_root / ".venv" / "Lib" / "site-packages"
if local_site_pkgs.exists() and str(local_site_pkgs) not in sys.path:
    sys.path.insert(0, str(local_site_pkgs))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

POSSIBLE_PATHS = [
    repo_root / "data" / "processed" / "model_b_augmented.jsonl",
    repo_root / "data" / "processed" / "model_b_clean.jsonl",
    repo_root / "data" / "raw" / "kwismo_data" / "messages.jsonl",
    Path("/content/drive/MyDrive/kwismo_data/messages.jsonl"),
    Path("/content/drive/MyDrive/messages.jsonl")
]

dataset_path = None
for p in POSSIBLE_PATHS:
    if p.exists():
        dataset_path = p
        break

if not dataset_path:
    for search_root in [repo_root, Path('/content/drive/MyDrive'), Path('/content')]:
        if search_root.exists():
            found = list(search_root.rglob("*.jsonl"))
            if found:
                dataset_path = found[0]
                break

print(f"Dataset chargé pour l'analyse visuelle : {dataset_path}")

## 2. Chargement des Données & Vue d'Ensemble

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Style graphique moderne
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 10

records = []
if dataset_path and dataset_path.is_file():
    with open(dataset_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Total d'enregistrements : {len(df)}")
display(df.head(3))

## 3. Graphique 1 : Distribution des Longueurs de Textes (Mots & Caractères)

In [ ]:
text_col = None
for c in ["texte", "text", "description", "content"]:
    if c in df.columns:
        text_col = c
        break

if text_col:
    df["longueur_caracteres"] = df[text_col].fillna("").apply(len)
    df["nombre_mots"] = df[text_col].fillna("").apply(lambda t: len(t.split()))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogramme 1 : Nombre de mots
    sns.histplot(df["nombre_mots"], kde=True, color="#2E86AB", ax=ax1, bins=25)
    ax1.set_title("Distribution du Nombre de Mots par Message", fontsize=12, fontweight="bold")
    ax1.set_xlabel("Nombre de mots")
    ax1.set_ylabel("Fréquence")
    
    # Histogramme 2 : Nombre de caractères
    sns.histplot(df["longueur_caracteres"], kde=True, color="#A23B72", ax=ax2, bins=25)
    ax2.set_title("Distribution de la Longueur en Caractères", fontsize=12, fontweight="bold")
    ax2.set_xlabel("Nombre de caractères")
    ax2.set_ylabel("Fréquence")
    
    plt.tight_layout()
    plt.show()

## 4. Graphique 2 : Top 15 des Termes les Plus Fréquents dans les Signalements

In [ ]:
from collections import Counter
import re

if text_col:
    all_words = []
    stopwords = {"de", "la", "le", "les", "des", "un", "une", "et", "a", "en", "du", "pour", "sur", "est", "pas", "plus", "par", "que", "dans", "avec", "au", "ce", "qui", "ne", "http", "https", "com"}
    
    for text in df[text_col].dropna():
        words = re.findall(r"\b\w+\b", str(text).lower())
        filtered = [w for w in words if w not in stopwords and len(w) > 2 and not w.isdigit()]
        all_words.extend(filtered)
        
    counter = Counter(all_words)
    top15 = counter.most_common(15)
    
    words_df = pd.DataFrame(top15, columns=["Mot", "Fréquence"])
    
    plt.figure(figsize=(12, 6))
    sns.barplot(data=words_df, x="Fréquence", y="Mot", palette="viridis")
    plt.title("Top 15 des Mots les Plus Fréquents (Hors Stopwords)", fontsize=13, fontweight="bold")
    plt.xlabel("Nombre d'occurrences")
    plt.ylabel("Mots clés")
    plt.tight_layout()
    plt.show()

## 5. Graphique 3 : Répartition par Type de Données (Texte vs Image OCR)

In [ ]:
type_col = None
for c in ["type_origine", "type", "content_type"]:
    if c in df.columns:
        type_col = c
        break

if type_col:
    type_counts = df[type_col].value_counts()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # Camembert / Donut chart
    colors = ["#2A9D8F", "#E76F51", "#F4A261", "#E9C46A"]
    ax1.pie(type_counts, labels=type_counts.index, autopct="%1.1f%%", startangle=140, colors=colors[:len(type_counts)], wedgeprops=dict(width=0.4, edgecolor='w'))
    ax1.set_title("Répartition en Donut Chart (Format d'origine)", fontsize=11, fontweight="bold")
    
    # Diagramme en barres
    sns.barplot(x=type_counts.index, y=type_counts.values, ax=ax2, palette="crest")
    ax2.set_title("Volume par Type de Média", fontsize=11, fontweight="bold")
    ax2.set_ylabel("Nombre d'extraits")
    
    plt.tight_layout()
    plt.show()

## 6. Graphique 4 : Distribution des Catégories Détectées par l'IA

In [ ]:
from src.models.model_b.preprocess import categorize_description

if text_col:
    df["categorie_ia"] = df[text_col].dropna().apply(categorize_description)
    cat_counts = df["categorie_ia"].value_counts().reset_index()
    cat_counts.columns = ["Catégorie", "Total"]
    
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(data=cat_counts, x="Total", y="Catégorie", palette="Spectral")
    plt.title("Répartition des Catégories d'Escroqueries & Contenu Légitime", fontsize=13, fontweight="bold")
    plt.xlabel("Nombre de signalements")
    plt.ylabel("Catégorie IA")
    
    # Ajout des étiquettes de valeurs
    for p in ax.patches:
        width = p.get_width()
        ax.annotate(f"{int(width)}", (width + 1, p.get_y() + p.get_height() / 2.), ha="left", va="center")
        
    plt.tight_layout()
    plt.show()